### In this notebook we show how the model to model evaluation module works

In [1]:
import sys, json

sys.path.append("../")
sys.path.append("../model_evaluation/")

In [2]:
# load some examples from examples folder in Signavio json format
filename_ground_truth = f"../examples/misc_booking_flight_tickets.json"
with open(filename_ground_truth, "r") as infile:
    model_1 = json.load(infile)


filename_generated = f"../examples/misc_loan_brokerage.json"
with open(filename_generated, "r") as infile:
    model_2 = json.load(infile)

In [3]:
import json
from BPMN_conversion import BPMNConverter


model_1_json = json.loads(BPMNConverter.convert(model_1).to_json())
# misc_bft_json_original = original_BPMNConverter.convert(E4_1).to_json()

model_2_json = json.loads(BPMNConverter.convert(model_2).to_json())

In [4]:
# ==============================================================================
# BPMN Model Comparison Pipeline
# ==============================================================================
import json
from rendering import create_similarity_dashboard, print_similarity_report
from bpmn_normalization import normalize_atomic_names
from bpmn_similarity import calculate_bpmn_similarity
from utils import cosine_sim_optimized


print("BPMN MODEL COMPARISON PIPELINE")


# Step 1: Model Summary
print("\n[1] MODEL STATISTICS")


def count_elements(model):
    """Count BPMN elements in a model."""
    return {
        "activities": len(model.get("activities", [])),
        "events": len(model.get("events", [])),
        "gateways": len(model.get("gateways", [])),
        "sequence_flows": len(model.get("sequenceFlows", [])),
        "message_flows": len(model.get("messageFlows", [])),
        "pools": len(model.get("pools", [])),
        "lanes": sum(len(p.get("lanes", [])) for p in model.get("pools", [])),
    }


model1_counts = count_elements(model_1_json)
model2_counts = count_elements(model_2_json)

print(f"Model 1: {sum(model1_counts.values())} total elements")
for key, val in model1_counts.items():
    if val > 0:
        print(f"  • {key.replace('_', ' ').title()}: {val}")

print(f"\nModel 2: {sum(model2_counts.values())} total elements")
for key, val in model2_counts.items():
    if val > 0:
        print(f"  • {key.replace('_', ' ').title()}: {val}")

# Step 2: Normalize Names
print("\n[2] SEMANTIC NAME NORMALIZATION")

threshold = 0.6
print(f"Aligning element names using a sentence transformer model (threshold={threshold})...")

model2_aligned, mappings = normalize_atomic_names(model_1_json, model_2_json, cosine_sim_optimized, threshold=threshold)

total_mappings = sum(len(v) for v in mappings.values())
if total_mappings > 0:
    print(f"✓ Applied {total_mappings} semantic name mappings")
    for elem_type, mapping in mappings.items():
        if mapping:
            print(f"  • {elem_type}: {len(mapping)} mappings")
            # Show first example
            first_old, first_new = next(iter(mapping.items()))
            print(f"    Example: '{first_old}' → '{first_new}'")
else:
    print("✓ No mappings needed (names already aligned)")

# Step 3: Calculate Similarity Without Normalization
print("\n[3] SIMILARITY ANALYSIS")


similarity_without_norm = calculate_bpmn_similarity(model_1_json, model_2_json, method="dice")

similarity_with_norm = calculate_bpmn_similarity(model_1_json, model2_aligned, method="dice")

print("WITHOUT normalization:")
print(f"  Overall Similarity: {similarity_without_norm['overall']:.1%}")
for cat in ["structural", "flows", "organizational", "subprocess"]:
    score = similarity_without_norm["high_level_scores"][cat]
    print(f"    • {cat.title()}: {score:.1%}")

print("\nWITH normalization:")
print(f"  Overall Similarity: {similarity_with_norm['overall']:.1%}")
for cat in ["structural", "flows", "organizational", "subprocess"]:
    score = similarity_with_norm["high_level_scores"][cat]
    print(f"    • {cat.title()}: {score:.1%}")

improvement = similarity_with_norm["overall"] - similarity_without_norm["overall"]
print(f"\n  → Improvement: {improvement:+.1%} ({abs(improvement)*100:.1f} percentage points)")

# Store results for dashboard
similarity_results = similarity_with_norm


print("Pipeline complete. Results stored in 'similarity_results'.")

BPMN MODEL COMPARISON PIPELINE

[1] MODEL STATISTICS
Model 1: 45 total elements
  • Activities: 4
  • Events: 11
  • Gateways: 3
  • Sequence Flows: 15
  • Message Flows: 7
  • Pools: 3
  • Lanes: 2

Model 2: 54 total elements
  • Activities: 5
  • Events: 11
  • Gateways: 4
  • Sequence Flows: 18
  • Message Flows: 9
  • Pools: 4
  • Lanes: 3

[2] SEMANTIC NAME NORMALIZATION
Aligning element names using a sentence transformer model (threshold=0.6)...
✓ Applied 18 semantic name mappings
  • activity_names: 1 mappings
    Example: 'fill out a loan request' → 'Select the best offer and request tickets'
  • activity_types: 2 mappings
    Example: 'Task' → 'Task'
  • event_names: 5 mappings
    Example: 'received credit request' → 'Customer request processed'
  • event_types: 6 mappings
    Example: 'IntermediateMessageEventCatching' → 'IntermediateMessageEventCatching'
  • gateway_types: 2 mappings
    Example: 'Exclusive' → 'Exclusive'
  • pool_names: 2 mappings
    Example: 'customer' →

In [ ]:
import json
from rendering.dashboard import create_similarity_dashboard


# Create and display dashboard
dashboard = create_similarity_dashboard(
    model_1_json,
    model_2_json,
    similarity_func=cosine_sim_optimized,
    calculate_similarity_func=calculate_bpmn_similarity,
    normalize_func=normalize_atomic_names,
    initial_threshold=0.5,
)
dashboard.display()

NameError: name 'bert_cosine_optimized' is not defined

In [ ]:
# Use the reusable XML embed function
import importlib
import rendering

importlib.reload(rendering)

# Load a BPMN XML file and render it
with open("../examples/student_project.xml", "r", encoding="utf-8") as f:
    xml_str = f.read()

# Navigated viewer enables zoom/pan
rendering.render_bpmn_xml_embed(xml_str, height_px=500, navigated=True)